In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Fase Gold - Tempo") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [15]:
spark.sql("DROP TABLE IF EXISTS gold_projeto.dim_tempo")

DataFrame[]

In [16]:
spark.sql("SHOW TABLES IN gold_projeto").show()
print("-------------------------\n")
spark.sql("SHOW TABLES IN projeto").show()

+------------+------------+-----------+
|   namespace|   tableName|isTemporary|
+------------+------------+-----------+
|gold_projeto|dim_conteudo|      false|
+------------+------------+-----------+

-------------------------

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  projeto|        amazon_books|      false|
|  projeto|      amazon_credits|      false|
|  projeto|       amazon_titles|      false|
|  projeto|    amazon_titles_v2|      false|
|  projeto|attributesandpopu...|      false|
|  projeto|    book_movie_adapt|      false|
|  projeto|          box_office|      false|
|  projeto|        global_music|      false|
|  projeto|      grammy_winners|      false|
|  projeto|imdb_movie_adapta...|      false|
|  projeto|imdb_tvshow_adapt...|      false|
|  projeto|     netflix_credits|      false|
|  projeto|      netflix_titles|      false|
|  projeto|   netflix_titles_v2|      false|
|  proj

In [17]:
spark.sql("""
    CREATE EXTERNAL TABLE gold_projeto.dim_tempo (
        id_data INT,
        data STRING,
        day INT,
        month INT,
        year INT,
        month_name STRING,
        trimester INT,
        semester INT
    )
    USING DELTA
    LOCATION 'hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_tempo/'
""")

DataFrame[]

In [18]:
rotten_dates = spark.table("projeto.streaming_rating") \
                    .select(F.col("in_theaters_date").alias("data_raw")) \
                    .union(spark.table("projeto.streaming_rating").select(F.col("on_streaming_date")))

oscar_dates = spark.table("projeto.oscar_awards") \
                   .select(F.to_date(F.concat(F.col("award_year"), F.lit("-01-01"))).alias("data_raw"))

grammy_dates = spark.table("projeto.grammy_winners") \
                    .select(F.to_date(F.concat(F.col("year"), F.lit("-01-01"))).alias("data_raw"))

box_office_dates = spark.table("projeto.box_office") \
                        .select(F.to_date(F.concat(F.col("Year"), F.lit("-01-01"))).alias("data_raw"))

soundtrack_dates = spark.table("projeto.soundtracks") \
                        .select(F.to_date(F.concat(F.col("year"), F.lit("-01-01"))).alias("data_raw"))

streaming_dates = spark.table("projeto.streaming_titles") \
                       .select(F.to_date(F.concat(F.col("release_year"), F.lit("-01-01"))).alias("data_raw"))

print(f"Datas Rotten Tomatoes: {rotten_dates.count()}")
print(f"Datas Oscars: {oscar_dates.count()}")
print(f"Datas Grammys: {grammy_dates.count()}")
print(f"Datas Box Office: {box_office_dates.count()}")
print(f"Datas Soundtracks: {soundtrack_dates.count()}")
print(f"Datas Streaming Titles: {streaming_dates.count()}")

Datas Rotten Tomatoes: 32166
Datas Oscars: 18905
Datas Grammys: 25370
Datas Box Office: 8145
Datas Soundtracks: 3133
Datas Streaming Titles: 15717


In [19]:
df_datas_unicas = rotten_dates.union(oscar_dates) \
                              .union(grammy_dates) \
                              .union(box_office_dates) \
                              .union(soundtrack_dates) \
                              .union(streaming_dates) \
                              .filter("data_raw IS NOT NULL") \
                              .distinct()

print("-" * 30)
print(f"Total de Datas Únicas Consolidadas: {df_datas_unicas.count()}")
print("-" * 30)

df_datas_unicas.orderBy("data_raw").show(10)

------------------------------
Total de Datas Únicas Consolidadas: 7084
------------------------------
+----------+
|  data_raw|
+----------+
|1912-01-01|
|1914-01-01|
|1914-06-01|
|1915-01-01|
|1915-03-03|
|1916-01-01|
|1916-09-05|
|1917-01-01|
|1918-01-01|
|1919-01-01|
+----------+
only showing top 10 rows



In [20]:
windowSpec = Window.orderBy("data_raw")

df_with_id = df_datas_unicas.withColumn("id_data", F.row_number().over(windowSpec))

In [21]:
df_dim_tempo_final = df_with_id.select(
    F.col("id_data"),
    F.col("data_raw").alias("data"),
    F.dayofmonth("data_raw").alias("day"),
    F.month("data_raw").alias("month"),
    F.year("data_raw").alias("year"),
    F.date_format("data_raw", "MMMM").alias("month_name"),
    F.quarter("data_raw").alias("trimester"),
    F.when(F.month("data_raw") <= 6, 1).otherwise(2).alias("semester")
)

In [22]:
df_dim_tempo_final.show()

+-------+----------+---+-----+----+----------+---------+--------+
|id_data|      data|day|month|year|month_name|trimester|semester|
+-------+----------+---+-----+----+----------+---------+--------+
|      1|1912-01-01|  1|    1|1912|   January|        1|       1|
|      2|1914-01-01|  1|    1|1914|   January|        1|       1|
|      3|1914-06-01|  1|    6|1914|      June|        2|       1|
|      4|1915-01-01|  1|    1|1915|   January|        1|       1|
|      5|1915-03-03|  3|    3|1915|     March|        1|       1|
|      6|1916-01-01|  1|    1|1916|   January|        1|       1|
|      7|1916-09-05|  5|    9|1916| September|        3|       2|
|      8|1917-01-01|  1|    1|1917|   January|        1|       1|
|      9|1918-01-01|  1|    1|1918|   January|        1|       1|
|     10|1919-01-01|  1|    1|1919|   January|        1|       1|
|     11|1919-05-13| 13|    5|1919|       May|        2|       1|
|     12|1919-10-03|  3|   10|1919|   October|        4|       2|
|     13|1

In [23]:
df_dim_tempo_final.select("id_data", "data", "day", "month", "year", "month_name", "trimester", "semester") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save("hdfs://hdfs-nn:9000/gold/gold_projeto.db/dim_tempo")

In [24]:
spark.sql(
    """
    DESCRIBE HISTORY gold_projeto.dim_tempo
    """
).show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-12-22 18:31:...|  null|    null|       WRITE|{mode -> Overwrit...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 1, n...|        null|Apache-Spark/3.4....|
|      0|2025-12-22 18:30:...|  null|    null|CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|                  {}|        null|Apache-Spark/3.4.

In [25]:
spark.sql(
    """
    SELECT * FROM gold_projeto.dim_tempo
    """
).show()

+-------+----------+---+-----+----+----------+---------+--------+
|id_data|      data|day|month|year|month_name|trimester|semester|
+-------+----------+---+-----+----+----------+---------+--------+
|      1|1912-01-01|  1|    1|1912|   January|        1|       1|
|      2|1914-01-01|  1|    1|1914|   January|        1|       1|
|      3|1914-06-01|  1|    6|1914|      June|        2|       1|
|      4|1915-01-01|  1|    1|1915|   January|        1|       1|
|      5|1915-03-03|  3|    3|1915|     March|        1|       1|
|      6|1916-01-01|  1|    1|1916|   January|        1|       1|
|      7|1916-09-05|  5|    9|1916| September|        3|       2|
|      8|1917-01-01|  1|    1|1917|   January|        1|       1|
|      9|1918-01-01|  1|    1|1918|   January|        1|       1|
|     10|1919-01-01|  1|    1|1919|   January|        1|       1|
|     11|1919-05-13| 13|    5|1919|       May|        2|       1|
|     12|1919-10-03|  3|   10|1919|   October|        4|       2|
|     13|1

In [26]:
spark.stop()